# Debtor Analysis

## Objective

Analyze debtor performance to identify high-value debtors, collection efficiency, and outstanding liabilities.

### Business Questions

- Who are the highest-value debtors?
- Which debtors have the largest outstanding balances?
- Which debtors have the best collection rate?
- Which debtor types contribute the most recovery value?
- Which debtors should be prioritized for recovery efforts?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from src.preprocessing import prepare_data

pd.set_option("display.max_columns", None)

In [29]:
df = prepare_data()

c:\Users\Subhan\OneDrive\Desktop\Road Towards Goals\Reownlogics\Day-1\Insurance_Claims_Analysisdata\notebooks\src\preprocessing.py:80: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [30]:
debtor_summary = (
    df.groupby(["Debtor Name", "Debtor Type"])
      .agg(
          Claims=("Claim ID", "count"),
          Recovery=("Recovery Amount", "sum"),
          Collected=("Collected Amount", "sum"),
          Remaining=("Remaining Amount", "sum")
      )
      .reset_index()
)

debtor_summary["Collection Rate"] = (
    debtor_summary["Collected"]
    / debtor_summary["Recovery"]
    * 100
).round(2)

debtor_summary["Avg Recovery per Claim"] = (
    debtor_summary["Recovery"]
    / debtor_summary["Claims"]
).round(2)

debtor_summary.sort_values(
    "Recovery",
    ascending=False,
    inplace=True
)

debtor_summary.head(20)

,Debtor Name,Debtor Type,Claims,Recovery,Collected,Remaining,Collection Rate,Avg Recovery per Claim
1127,ساجر زامن رادا كام,insured,2,325588.00,0.0,325588.00,0.00,162794.00
215,SHOUKHATHALI NARAKKODAN,insured,1,325230.00,0.0,325230.00,0.00,325230.00
2908,ناشي غازي العتيبي,insured,1,314630.00,0.0,314630.00,0.00,314630.00
2831,منصور حسين الجعفري,insured,1,300000.00,0.0,300000.00,0.00,300000.00
2541,محمد عابد الزويهري,insured,1,300000.00,0.0,300000.00,0.00,300000.00
1111,زياد مسفر الزهراني,third_party,1,156101.40,20000.0,136101.00,12.81,156101.40
2857,منيره سليمان الحربي,insured,1,153533.33,0.0,153533.33,0.00,153533.33
2429,محمد الدويك,insured,1,139031.00,0.0,139031.00,0.00,139031.00
475,الاميره البندري بن هذلول ال سعود,insured,1,130550.00,0.0,130550.00,0.00,130550.00
50,AMER MOHAMMED BIN ABDULLAH ALOTAIBI,insured,1,115567.00,0.0,115567.00,0.00,115567.00


### Q1) Which debtors owe the most?

In [31]:
fig = px.bar(
    debtor_summary.sort_values("Recovery").tail(10),
    x="Recovery",
    y="Debtor Name",
    orientation="h",
    color="Recovery",
    text_auto=".2s",
    title="Top 10 Debtors by Recovery Amount"
)

fig.show()

### Q2) Which debtors have the highest outstanding balance?

In [32]:
fig = px.bar(
    debtor_summary.sort_values("Remaining").tail(10),
    x="Remaining",
    y="Debtor Name",
    orientation="h",
    color="Remaining",
    text_auto=".2s",
    title="Top 10  Outstanding Debtors "
)

fig.show()

### Q3) Which debtors pay most efficiently?

In [33]:
efficient_debtors = (
    debtor_summary[debtor_summary["Claims"] >= 2]
    .sort_values("Collection Rate", ascending=False)
    .head(20)
)

fig = px.bar(
    efficient_debtors.sort_values("Collection Rate"),
    x="Collection Rate",
    y="Debtor Name",
    orientation="h",
    color="Collection Rate",
    text="Collection Rate",
    title="Top Collection Rate by Debtor (Min. 2 Claims)"
)

fig.show()

### Q4) Which debtor type contributes most?

In [34]:
debtor_type_summary = (
    debtor_summary
    .groupby("Debtor Type")
    .agg(
        Debtors=("Debtor Name", "nunique"),
        Claims=("Claims", "sum"),
        Recovery=("Recovery", "sum"),
        Collected=("Collected", "sum"),
        Remaining=("Remaining", "sum")
    )
    .reset_index()
)

debtor_type_summary["Collection Rate"] = (
    debtor_type_summary["Collected"]
    / debtor_type_summary["Recovery"]
    * 100
).round(2)

debtor_type_summary

,Debtor Type,Debtors,Claims,Recovery,Collected,Remaining,Collection Rate
0,Insured,9,9,62365.95,4828.47,57537.48,7.74
1,insured,2770,2817,31980429.20,1136817.55,30843597.51,3.55
2,third_party,434,439,3691073.18,344779.62,3344292.06,9.34


In [35]:
fig = px.bar(
    debtor_type_summary,
    x="Debtor Type",
    y="Recovery",
    color="Debtor Type",
    text_auto=".2s",
    title="Recovery Amount by Debtor Type"
)

fig.show()

### Debtor Performance Matrix:
Q5) Which debtors should be prioritized for recovery efforts?

In [36]:
fig = px.scatter(
    debtor_summary[
        debtor_summary["Claims"] >= 3 #change this value to filter debtors 
    ],
    x="Recovery",
    y="Collection Rate",
    size="Claims",
    color="Remaining",
    hover_name="Debtor Name",
    title="Debtor Performance Matrix"
)

fig.show()

### Executive Insights

In [37]:
highest_recovery = debtor_summary.iloc[0]

highest_remaining = (
    debtor_summary
    .sort_values("Remaining", ascending=False)
    .iloc[0]
)

best_collector = (
    debtor_summary[debtor_summary["Claims"] >= 2]
    .sort_values("Collection Rate", ascending=False)
    .iloc[0]
)

print(f"""
📊 DEBTOR INSIGHTS

• Largest Debtor:
  {highest_recovery['Debtor Name']}
  SAR {highest_recovery['Recovery']:,.0f}

• Largest Outstanding:
  {highest_remaining['Debtor Name']}
  SAR {highest_remaining['Remaining']:,.0f}

• Highest Collection Rate:
  {best_collector['Debtor Name']}
  {best_collector['Collection Rate']:.2f}%
""")


📊 DEBTOR INSIGHTS

• Largest Debtor:
  ساجر زامن رادا كام
  SAR 325,588

• Largest Outstanding:
  ساجر زامن رادا كام
  SAR 325,588

• Highest Collection Rate:
  محمد عايض محمد الشمراني
  100.00%



In [38]:
priority_debtors = debtor_summary[
    (debtor_summary["Recovery"] > debtor_summary["Recovery"].median()) &
    (debtor_summary["Collection Rate"] < 20)
].sort_values("Remaining", ascending=False)

priority_debtors[
    [
        "Debtor Name",
        "Recovery",
        "Remaining",
        "Collection Rate"
    ]
]

,Debtor Name,Recovery,Remaining,Collection Rate
1127,ساجر زامن رادا كام,325588.0,325588.0,0.0
215,SHOUKHATHALI NARAKKODAN,325230.0,325230.0,0.0
2908,ناشي غازي العتيبي,314630.0,314630.0,0.0
2831,منصور حسين الجعفري,300000.0,300000.0,0.0
2541,محمد عابد الزويهري,300000.0,300000.0,0.0
...,...,...,...,...
1196,سعد عبدالله اللحيدان,8030.0,8030.0,0.0
2937,نامي منيع المطيري,8030.0,8030.0,0.0
2774,مشعل متعب العتيبي,8021.0,8021.0,0.0
2396,محمد أحمد محمد عداوي,8018.0,8018.0,0.0


# Conclusion

This analysis identifies debtors with the highest financial exposure, evaluates collection efficiency, and highlights accounts requiring immediate recovery attention.

The findings support data-driven prioritization of recovery activities and will be integrated into the executive dashboard.